# CA Experiment 8 — Dynamic-Memory Self-Model + Re-Grounded E

**Runtime:** Colab Pro **A100** + local MiniLM embeddings (free) + bounded Anthropic API (Sonnet 4.6 multi-session scripts + disclaim data, Sonnet 4.5 judge). Reuses the Exp 6/7 checkpoints; the one training step is a short **Stage-C re-fit** that *removes* the baked episodic fixtures → `sft_ada_dynamic`.

Exp 6 reached its Episodic (E) score by **baking three fictional `salient_past_events`** into the weights — false day-1 memories. Exp 8 removes them, replaces them with a **live multi-session Episodic Register**, and re-grounds E on *dynamic recall via extract-then-style* (Exp 7's span path), including the day-1 case where the correct answer is "we haven't spoken before."

**Pipeline**
```
gen multi-session scripts (cross-session anchors + day-1 probes)
  + gen day-1 disclaim SFT data (honest 'no record yet')
  -> build re-fit set (drop recall, scrub baked events, swap self-model, add disclaim)
  -> Stage-C re-fit  ->  sft_ada_dynamic
  -> run B0 (baked, control) / D0 (dynamic, register off) / D1 (dynamic, register on)
  -> judge reliability -> analyse (false-memory, recall-vs-gap, persona, cost)
```
See `CA_Experiment8_Plan.md`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/CA_Experiment_8'
EXP6_DIR    = '/content/drive/MyDrive/CA_Experiment_6'
EXP7_DIR    = '/content/drive/MyDrive/CA_Experiment_7'
BAKED_CKPT  = f'{EXP6_DIR}/checkpoints/sft_ada_final.pt'          # Exp 6 baked model (B0)
DYN_CKPT    = f'{EXP6_DIR}/checkpoints/sft_ada_dynamic_final.pt'  # produced by the re-fit below (D0/D1)
SPAN_CKPT   = f'{EXP6_DIR}/checkpoints/span_final.pt'            # extractive recall head
TOKENIZER   = f'{EXP6_DIR}/tokenizer/ada_bpe.json'
PRETRAIN_BIN = f'{EXP6_DIR}/data/pretrain4b/train.bin'          # replay mix (optional)
assert os.path.exists(PROJECT_DIR), f'Upload CA_Experiment_8 to Drive: {PROJECT_DIR}'
assert os.path.exists(BAKED_CKPT), f'Exp 6 checkpoint not found: {BAKED_CKPT}'
assert os.path.exists(SPAN_CKPT),  f'span head not found: {SPAN_CKPT}'

os.chdir(PROJECT_DIR)
for d in (PROJECT_DIR, EXP7_DIR, EXP6_DIR):
    if d not in sys.path:
        sys.path.insert(0, d)
# Export paths so BOTH $VAR (shell, in ! cells) and {VAR} (python) resolve.
for _k in ('PROJECT_DIR', 'EXP6_DIR', 'EXP7_DIR', 'BAKED_CKPT', 'DYN_CKPT', 'SPAN_CKPT', 'TOKENIZER', 'PRETRAIN_BIN'):
    os.environ[_k] = eval(_k)


def _parse_env(path):
    """Tolerant .env: BOM, blanks, comments, `export `, quotes."""
    for raw in open(path, encoding='utf-8-sig'):
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        if line.startswith('export '):
            line = line[len('export '):]
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip().strip('\'"'))


for _envp in (Path(PROJECT_DIR) / '.env', Path(EXP6_DIR) / '.env'):
    if _envp.exists():
        _parse_env(_envp)
print('cwd:', os.getcwd(), '| baked+span ok | dyn ckpt exists:', os.path.exists(DYN_CKPT))

In [ ]:
!pip install -q sentence-transformers anthropic python-dotenv 'tokenizers<=0.23.0' vaderSentiment qiskit qiskit-aer matplotlib pandas

# GPU Check

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> A100.')
p = torch.cuda.get_device_properties(0)
print('GPU  :', p.name, f'| VRAM {p.total_memory/1e9:.0f} GB | bf16 {torch.cuda.is_bf16_supported()}')

In [ ]:
# Sanity: dynamic compact SCI fits the 1024 window with room for a memory block, and carries NO baked events
from compact_sci_dynamic import build_compact_system_prompt_dynamic
from tokenizer_util import ADATokenizer

tok = ADATokenizer.load(TOKENIZER)
cs = build_compact_system_prompt_dynamic()
n = len(tok.encode(cs))
print(f'dynamic compact SCI: {n} tokens  (full baked SCI is 1338; target ~420)')
assert n < 700, 'compact SCI too large — no room for a memory block'
for bad in ('salient_past_events', '3422', 'Fermi', 'tungsten', 'election'):
    assert bad.lower() not in cs.lower(), f'baked event leaked into the dynamic SCI: {bad}'
print('OK: dispositional-only, no baked events\n')
print(cs[:500], '...')

# API Check
Live Anthropic round-trip; aliases `CHA_EXPERIMENT_SONNET_KEY` → `ANTHROPIC_API_KEY` so bare `anthropic.Anthropic()` works everywhere (script gen + judges).

In [ ]:
GEN_MODEL   = 'claude-sonnet-4-6'   # multi-session scripts + disclaim data
JUDGE_MODEL = 'claude-sonnet-4-5'   # dynamic-E + PersonaScore judge (matches Exp 1-7)
os.environ['GEN_MODEL'] = GEN_MODEL

_key = os.environ.get('CHA_EXPERIMENT_SONNET_KEY') or os.environ.get('ANTHROPIC_API_KEY')
if _key:
    os.environ['ANTHROPIC_API_KEY'] = _key   # so bare anthropic.Anthropic() works everywhere too
    import anthropic
    r = anthropic.Anthropic(api_key=_key).messages.create(model=JUDGE_MODEL, max_tokens=10,
        messages=[{'role': 'user', 'content': 'Say "ok".'}])
    print('API ok:', r.content[0].text.strip(), f'(gen={GEN_MODEL} judge={JUDGE_MODEL})')
else:
    print('No API key - set CHA_EXPERIMENT_SONNET_KEY (or ANTHROPIC_API_KEY) in .env')

## Step 1 — multi-session scripts
Several sessions per user with cross-session anchors (recall tagged by `session_gap`) + **day-1 probes** (asked before anything is stored — correct answer is to disclaim). Fillers come free from the Exp 6 `persona_scripts`; Sonnet spends only on the distinctive anchors.

In [ ]:
!python gen_multisession_scripts.py --n-users 6 --sessions-per-user 4 --turns-per-session 25 \
  --anchors-per-session 2 --day1-probes 3 --max-recall-gaps 3 --n-anchors 24 --model $GEN_MODEL

## Step 2 — day-1 disclaim SFT data
The positive signal for H1: in-character "I have no record of prior sessions yet" over varied past-session probes with no provided record. Rendered with the **dynamic** self-model (no events).

In [ ]:
!python gen_disclaim_data.py --budget 12 --pairs-per-call 8 --model $GEN_MODEL \
  --out data/qa_sft_disclaim.jsonl

## Step 3 — build the re-fit SFT set
Transform the Exp 6 SFT set: **swap the self-model** on every kept record → dynamic (no events, ever); **drop** `sonnet_recall`/`sonnet_memrecall`; **scrub** records whose assistant turns recite a baked fixture; **append** the disclaim data. The event-leak guard must report 0.

In [ ]:
!python build_refit_sft.py --in $EXP6_DIR/data/qa_sft.jsonl \
  --disclaim data/qa_sft_disclaim.jsonl --out data/qa_sft_dynamic.jsonl

## Step 4 — Stage-C re-fit → `sft_ada_dynamic`
Short re-fit from `sft_ada` on the dynamic set (~350 steps, low LR) — un-bakes the events without the Exp 7 Phase-B overfit. Replay-mixes Stage-A LM data to hold fluency. Writes `sft_ada_dynamic_final.pt` to the Exp 6 checkpoints dir.

In [ ]:
replay = f'--pretrain-bin {PRETRAIN_BIN}' if os.path.exists(PRETRAIN_BIN) else ''
print('replay:', replay or '(none — pretrain bin absent; re-fit without LM replay)')
!python $EXP6_DIR/train_sft.py --init $BAKED_CKPT --sft data/qa_sft_dynamic.jsonl {replay} \
  --tokenizer $TOKENIZER --ckpt-dir $EXP6_DIR/checkpoints --run-name sft_ada_dynamic \
  --max-steps 350 --batch-size 16 --grad-accum 4 --lr 5e-5 --replay-frac 0.1 --prefix-lm
assert os.path.exists(DYN_CKPT), f're-fit did not produce {DYN_CKPT}'
print('re-fit done →', DYN_CKPT)

## Step 5 — pilot (no API, no MiniLM)
Validate the full D1 harness cheaply before the judged run: dry judge + stub embedder, 1 script.

In [ ]:
!python evaluate_dynamic_e.py --condition D1 --dynamic-checkpoint $DYN_CKPT \
  --span-checkpoint $SPAN_CKPT --limit 1 --stub-embedder --dry-run-judge --out-dir results_pilot

## Step 6 — run B0 / D0 / D1
B0 = baked model (fabricates day-1, the control); D0 = dynamic model, register off (honest but forgetful); D1 = dynamic model, register on (discloses day-1, recalls later).

In [ ]:
for cond in ('B0', 'D0', 'D1'):
    print('=' * 44, cond)
    !python evaluate_dynamic_e.py --condition {cond} \
      --baked-checkpoint $BAKED_CKPT --dynamic-checkpoint $DYN_CKPT --span-checkpoint $SPAN_CKPT \
      --memory-budget 300 --top-k 5 --window-n 8 --persona-interval 10 --resume

## Step 7 — judge reliability (κ_w ≥ 0.70 gate)
Re-score a 5% sample of T/C/S persona probes and the dynamic-E probes at T=0; weighted Cohen's κ against the logged scores (Exp 1-7 gate).

In [ ]:
import json, random, glob
from ca_assets import cohens_kappa
from evaluate import persona_judge
from evaluate_dynamic_e import _dynamic_e_judge

rows = []
for p in glob.glob('results/D1/scores_*.jsonl'):
    rows += [json.loads(l) for l in open(p) if l.strip()]
random.seed(1337)

persona = [r for r in rows if r['kind'] == 'persona']
if persona:
    s = random.sample(persona, max(1, int(len(persona) * 0.05)))
    a = [r['score'] for r in s]
    b = [persona_judge(r['probe'], r['response'], r['dimension'])[0] for r in s]
    print('persona  κ_w', round(cohens_kappa(a, b, weighted=True), 3), '| n', len(s),
          '| gate', cohens_kappa(a, b, weighted=True) >= 0.70)

epi = [r for r in rows if r['kind'] in ('day1', 'recall')]
if epi:
    s = random.sample(epi, max(1, int(len(epi) * 0.05)))
    a = [r['score'] for r in s]
    b = [_dynamic_e_judge(r['probe'], r['expected'], r['response'], r['kind'] == 'day1')[0] for r in s]
    print('dynamic-E κ_w', round(cohens_kappa(a, b, weighted=True), 3), '| n', len(s),
          '| gate', cohens_kappa(a, b, weighted=True) >= 0.70)

## Step 8 — analyse + figures
False-memory rate (H1), dynamic recall vs session gap (H2/H4), re-grounded E vs the baked-fixture E (3.29), persona T/C/S (H3), cost. Prints the pre-registered decision table.

In [ ]:
!python analyse_results.py --results-dir results

from IPython.display import Image, display
for f in ('exp8_false_memory', 'exp8_recall_vs_gap', 'exp8_E_persona'):
    p = f'results/{f}.png'
    if os.path.exists(p):
        print('\n===', f, '==='); display(Image(filename=p))

In [ ]:
import json
import pandas as pd
out = json.load(open('results/analysis_data.json'))
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

print('=' * 70, '\nH1 — day-1 false-memory (baked B0 vs dynamic D0/D1)')
display(pd.DataFrame([{'cond': c, **out['false_memory'][c]} for c in ('B0', 'D0', 'D1') if c in out['false_memory']]))

print('H2/H4 — dynamic recall (D1) by session gap')
if 'D1' in out['recall']:
    rc = out['recall']['D1']
    display(pd.DataFrame([{'session_gap': g, **v} for g, v in rc['by_gap'].items()]))
    print(f"  overall recall acc {rc['recall_acc']:.3f} | fabricate {rc['fabricate_rate']:.3f}")

print('\nRe-grounded E + persona T/C/S')
display(pd.DataFrame([{'cond': c,
                       'overall_E': out['episodic_E'].get(c, {}).get('overall_E'),
                       'vs_fixture_E(3.29)': out['episodic_E'].get(c, {}).get('vs_exp6_fixture_E'),
                       'persona': out['persona'].get(c, {}).get('overall')}
                      for c in ('B0', 'D0', 'D1')]))

print('\nDECISION (pre-registered §3)')
print(json.dumps(out['decision'], indent=2))